# Real-to-synthetic overdraft detection

This notebook trains a **supervised rare-event classifier** on real Berka transactions and evaluates the unchanged model on held-out real data and ClavaDDPM synthetic transactions. It is an overdraft detector, not an unsupervised anomaly-scoring model.

## Target selection

The candidate analysis favors `trans.balance < 0` for this task: it is a direct indicator of an account solvency breach and is available for transaction-level evaluation at scale. `loan.status` is an explicit severe-debt outcome, but describes only 682 loan contracts and is not the transaction overdraft target. The target is `overdraft = (balance < 0)`.

`balance` is used **only** to construct the label and is excluded from all predictors. Transaction/account IDs are also excluded. The predictor set is `trans_date`, `trans_type`, `operation`, `amount`, `k_symbol`, and `bank`; the beneficiary `account` field is excluded because it is an account identifier rather than a generalizable transaction attribute.

The full raw real transaction file is used here (1,056,320 rows in the bundled data); the preprocessed ClavaDDPM input is a 20,000-row sample. The generated synthetic table is evaluated using its own `balance < 0` labels. Metrics on synthetic data measure how well a detector trained on real data transfers to the generated distribution; they are not, on their own, proof that the synthetic table is realistic or that an individual prediction is correct.

In [ ]:
import os
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


SEED = 42
TARGET = "overdraft"
NUMERIC_FEATURES = ["trans_date", "amount"]
CATEGORICAL_FEATURES = ["trans_type", "operation", "k_symbol", "bank"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

def find_project_root(start: Path | None = None) -> Path:
    """Find the repository root from the current directory or notebook path."""
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Could not find pyproject.toml in the current directory or its parents.")

ROOT = find_project_root()
BERKA_DIR = ROOT / "implementations/tabular_data/multi_table/data/berka"
RAW_TRANS_PATH = BERKA_DIR / "raw_data/trans.csv"
ENCODERS_PATH = BERKA_DIR / "label_encoders.pkl"
SYNTHETIC_TRANS_PATH = (
    ROOT
    / "implementations/tabular_data/multi_table/results/ClavaDDPM_notebook/trans/synthetic_final/trans_synthetic.csv"
)

for path in (RAW_TRANS_PATH, ENCODERS_PATH, SYNTHETIC_TRANS_PATH):
    if not path.is_file():
        raise FileNotFoundError(f"Required dataset/preprocessing artifact not found: {path}")

os.chdir(ROOT)
print(f"Project root: {ROOT}")
print(f"Real transactions: {RAW_TRANS_PATH}")
print(f"Synthetic transactions: {SYNTHETIC_TRANS_PATH}")

## Load and align real and synthetic data

The real table is transformed using the same project preprocessing helper and stored date epoch used by the multi-table pipeline. Synthetic discrete columns are decoded with the saved label encoders so the fitted one-hot encoder sees the same category names. Any generated category code outside the saved domain is retained as an explicit unknown category.

In [ ]:
from implementations.tabular_data.multi_table.data_preprocessing.pre_process_berka_all_tabels import (
    load_raw_table,
    preprocess_table,
)


with ENCODERS_PATH.open("rb") as file:
    preprocessing_artifacts = pickle.load(file)

saved_encoders = preprocessing_artifacts["encoders"]["trans"]
date_epoch = preprocessing_artifacts["date_epochs_yymmdd"]["trans"]
raw_real = load_raw_table(RAW_TRANS_PATH, "trans")
real_preprocessed, real_encoders, _ = preprocess_table("trans", raw_real, date_epoch)

for column in CATEGORICAL_FEATURES:
    saved_classes = saved_encoders[column].classes_
    fitted_classes = real_encoders[column].classes_
    if not np.array_equal(saved_classes, fitted_classes):
        raise ValueError(
            f"Category mapping mismatch for {column!r}; synthetic codes cannot be safely decoded.",
        )

real = real_preprocessed.copy()
real[TARGET] = (pd.to_numeric(real["balance"], errors="raise") < 0).astype("int8")
for column in CATEGORICAL_FEATURES:
    real[column] = real[column].astype(str)

synthetic = pd.read_csv(SYNTHETIC_TRANS_PATH)
required_synthetic_columns = set(FEATURES + ["balance"])
missing_synthetic = required_synthetic_columns.difference(synthetic.columns)
if missing_synthetic:
    raise ValueError(f"Synthetic transactions are missing required columns: {sorted(missing_synthetic)}")

for column in NUMERIC_FEATURES + ["balance"]:
    synthetic[column] = pd.to_numeric(synthetic[column], errors="coerce")
if synthetic["balance"].isna().any():
    raise ValueError("Synthetic balance contains missing/non-numeric values; target labels cannot be assigned.")

unknown_category_counts = {}
for column in CATEGORICAL_FEATURES:
    codes = pd.to_numeric(synthetic[column], errors="coerce")
    rounded_codes = codes.round()
    valid_codes = codes.notna() & np.isclose(codes, rounded_codes)
    decoded = pd.Series("__unknown__", index=synthetic.index, dtype="object")
    valid_codes &= rounded_codes.between(0, len(saved_encoders[column].classes_) - 1)
    if valid_codes.any():
        decoded.loc[valid_codes] = saved_encoders[column].inverse_transform(
            rounded_codes.loc[valid_codes].astype("int64"),
        )
    unknown_category_counts[column] = int((~valid_codes).sum())
    synthetic[column] = decoded

synthetic[TARGET] = (synthetic["balance"] < 0).astype("int8")

summary = pd.DataFrame(
    {
        "rows": [len(real), len(synthetic)],
        "overdrafts": [int(real[TARGET].sum()), int(synthetic[TARGET].sum())],
        "overdraft prevalence": [real[TARGET].mean(), synthetic[TARGET].mean()],
    },
    index=["real", "synthetic"],
)
display(summary)
print("Synthetic category values mapped to unknown:", unknown_category_counts)
print("Predictors:", FEATURES)

## Train on real transactions

The real train/validation/test partitions are grouped by `account_id`: transactions from one account cannot appear in multiple partitions. The ID is used only for splitting and never enters the model. A class-weighted random forest addresses label imbalance; all preprocessing is fitted inside the training pipeline.

In [ ]:
test_splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_val_idx, real_test_idx = next(test_splitter.split(real, real[TARGET], groups=real["account_id"]))
train_val = real.iloc[train_val_idx]
validation_splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED + 1)
train_idx, validation_idx = next(
    validation_splitter.split(
        train_val, train_val[TARGET], groups=train_val["account_id"],
    ),
)
real_train = train_val.iloc[train_idx]
real_validation = train_val.iloc[validation_idx]
real_test = real.iloc[real_test_idx]

if any(partition[TARGET].nunique() < 2 for partition in (real_train, real_validation, real_test)):
    raise ValueError("The account-grouped split must contain both target classes in every partition.")

numeric_pipeline = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="median"))],
)
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ],
)
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, NUMERIC_FEATURES),
        ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
    ],
)
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=150,
                max_depth=14,
                min_samples_leaf=10,
                class_weight="balanced_subsample",
                max_samples=0.5,
                n_jobs=-1,
                random_state=SEED,
            ),
        ),
    ],
)

model.fit(real_train[FEATURES], real_train[TARGET])
print(
    f"Train rows: {len(real_train):,}; validation rows: {len(real_validation):,}; "
    f"real test rows: {len(real_test):,}",
)
print(
    "Unique accounts - train/validation/test: "
    f"{real_train['account_id'].nunique():,}/"
    f"{real_validation['account_id'].nunique():,}/"
    f"{real_test['account_id'].nunique():,}",
)
print(
    "Positive rates - train/validation/test: "
    f"{real_train[TARGET].mean():.3%}/"
    f"{real_validation[TARGET].mean():.3%}/"
    f"{real_test[TARGET].mean():.3%}",
)

## Compare real holdout and synthetic transfer

Choose a decision threshold by maximizing F1 on the real validation partition only, then freeze it for both the held-out real test and synthetic evaluation. Average precision (area under the precision-recall curve) is emphasized because overdrafts are rare. ROC-AUC is included for comparison, alongside precision, recall, and F1 at the selected threshold.

In [ ]:
validation_probabilities = model.predict_proba(real_validation[FEATURES])[:, 1]
validation_precision, validation_recall, candidate_thresholds = precision_recall_curve(
    real_validation[TARGET], validation_probabilities,
)
validation_f1 = (
    2 * validation_precision[:-1] * validation_recall[:-1]
    / np.maximum(validation_precision[:-1] + validation_recall[:-1], 1e-12)
)
if candidate_thresholds.size == 0:
    raise ValueError("Could not select a threshold from the real validation predictions.")
THRESHOLD = float(candidate_thresholds[int(np.argmax(validation_f1))])
print(f"Validation-selected threshold (maximum F1): {THRESHOLD:.6f}")


def evaluate_dataset(name: str, data: pd.DataFrame) -> tuple[dict[str, float], np.ndarray, np.ndarray]:
    """Evaluate the real-trained detector on one labeled dataset."""
    y_true = data[TARGET].to_numpy(dtype="int8")
    if np.unique(y_true).size != 2:
        raise ValueError(f"{name} evaluation requires both overdraft and non-overdraft examples.")
    probabilities = model.predict_proba(data[FEATURES])[:, 1]
    predictions = (probabilities >= THRESHOLD).astype("int8")
    scores = {
        "rows": float(len(data)),
        "overdraft prevalence": float(y_true.mean()),
        "decision threshold": THRESHOLD,
        "average precision": float(average_precision_score(y_true, probabilities)),
        "ROC-AUC": float(roc_auc_score(y_true, probabilities)),
        f"precision @ {THRESHOLD:.3f}": float(
            np.sum((predictions == 1) & (y_true == 1)) / max(np.sum(predictions == 1), 1),
        ),
        f"recall @ {THRESHOLD:.3f}": float(
            np.sum((predictions == 1) & (y_true == 1)) / max(np.sum(y_true == 1), 1),
        ),
        f"F1 @ {THRESHOLD:.3f}": float(
            2
            * np.sum((predictions == 1) & (y_true == 1))
            / max(np.sum(predictions == 1) + np.sum(y_true == 1), 1),
        ),
    }
    print(f"{name}:\n{classification_report(y_true, predictions, target_names=['non-overdraft', 'overdraft'], zero_division=0)}")
    print("Confusion matrix [ [TN, FP], [FN, TP] ]:")
    print(confusion_matrix(y_true, predictions, labels=[0, 1]))
    return scores, y_true, probabilities

real_test_scores, y_real_test, p_real_test = evaluate_dataset("Real held-out accounts", real_test)
synthetic_scores, y_synthetic, p_synthetic = evaluate_dataset("Synthetic transactions", synthetic)

results = pd.DataFrame(
    [real_test_scores, synthetic_scores],
    index=["Real held-out", "Synthetic"],
).astype({"rows": "int64"})
display(results)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for name, y_true, probabilities in [
    ("Real held-out", y_real_test, p_real_test),
    ("Synthetic", y_synthetic, p_synthetic),
]:
    precision, recall, _ = precision_recall_curve(y_true, probabilities)
    ax.plot(recall, precision, label=name)
ax.set(title="Overdraft precision-recall curves", xlabel="Recall", ylabel="Precision")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.grid(alpha=0.3)
ax.legend()
plt.show()

## Interpretation and limitations

- Compare synthetic average precision and recall with the real account-held-out results; a large gap indicates transfer/domain shift, not necessarily a failure of the real-data model.
- Compare overdraft prevalences first. A synthetic label rate that differs from real data changes precision and average precision even if ranking quality is similar.
- The real and synthetic overdraft labels are both derived from `balance < 0`; `balance` is deliberately absent from `FEATURES`, so the target cannot be trivially read from the predictors.
- Inputs use the transaction table only. The supplied candidate table notes that richer account/client/district context can help, but adding joins and historical aggregates requires leakage-safe temporal feature engineering and aligned synthetic parent-child keys, which this first experiment does not claim to provide.
- This is a baseline experiment. If operational decisions require a different precision/recall trade-off, choose a threshold using a real validation partition, then freeze it before evaluating any test or synthetic set.